# Semantic Model Similarity

Compare every semantic model cataloged in the lakehouse and identify duplicates, near-duplicates, and subset relationships. Each model is reduced to a signature (tables, columns, measure names, measure definitions, DAX expressions, relationships, and data sources). Every pair gets two independent scores:

- a symmetric **composite similarity score** — structural Jaccard overlap plus text-embedding cosine similarity, tiered as duplicate / similar / distinct — that answers *"how alike are these two models overall?"*, and
- a directional **containment score** that answers *"does one model contain everything in the other?"*.

Duplicates are grouped into clusters. Results are shown in-notebook and written back to the lakehouse as Delta tables.


## Prerequisites

Run this notebook in a Fabric notebook runtime with a lakehouse attached — the same lakehouse the TOM catalog notebook wrote to. It must already contain the catalog Delta tables (`semantic_models`, `semantic_model_tables`, `semantic_model_columns`, `semantic_model_relationships`, `semantic_model_measures`, `semantic_model_datasources`). Text similarity uses a local scikit-learn TF-IDF vectorizer, so no external endpoint, key, or GPU/PyTorch runtime is required.

## Parameters

All tunable settings live here: the write mode, blocking toggle, tier thresholds, the ranked-table and heatmap knobs, and the per-signal weights. Adjust these, then run the notebook top to bottom. The catalog tables are read from the **attached lakehouse**, and results are written back to it.

In [ ]:
# Results are read from and written to the lakehouse attached to this notebook.
WRITE_MODE = "overwrite"  # "overwrite" replaces prior output; use "append" only if downstream supports it.

# Blocking limits comparisons to model pairs that share at least one table or measure name.
# Disable to force full pairwise comparison (slower on large catalogs).
ENABLE_BLOCKING = True

# Composite-score tier thresholds.
DUPLICATE_THRESHOLD = 0.95
SIMILAR_THRESHOLD = 0.70

# Containment threshold. Containment is a second, directional score that answers a
# different question than the composite similarity score: "does one model contain
# everything in the other?" rather than "how alike are the two models overall?".
# A pair whose stronger direction reaches this value is flagged as a containment
# candidate, and the two scores are reported side by side so you can filter on both.
CONTAINMENT_THRESHOLD = 0.95

# Report knobs.
TOP_N = 20  # Rows shown in the ranked pair table.
HEATMAP_MIN_SCORE = SIMILAR_THRESHOLD  # Hide heatmap cells scoring below this composite value.

# Relative weights for each similarity signal. Values are normalized, so they need not sum to 1.
SIMILARITY_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_dax_embedding": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Relative weights for each containment signal. Containment uses exact measure
# definitions (name + comment-stripped DAX) instead of the TF-IDF embedding, because
# lexical cosine similarity is symmetric and cannot establish that one model's measures
# are a subset of another's. Weights are normalized per direction over whichever signals
# the source model actually has.
CONTAINMENT_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_definitions": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Guard against misconfigured weights before any scoring runs.
for _weights_name, _weights in (
    ("SIMILARITY_WEIGHTS", SIMILARITY_WEIGHTS),
    ("CONTAINMENT_WEIGHTS", CONTAINMENT_WEIGHTS),
):
    if any(weight < 0 for weight in _weights.values()):
        raise ValueError(f"{_weights_name} must not contain negative weights.")
    if sum(_weights.values()) <= 0:
        raise ValueError(f"{_weights_name} must sum to a positive value.")


## Setup and computation

The code that builds the model signatures and scores every pair. Its input is hidden by default so you can focus on the parameters and results - click **Show input** on any cell to inspect it, or fold the whole section from this heading. You don't need to touch anything here.

In [ ]:
import itertools
import re
from collections import defaultdict

import numpy as np
import pandas as pd

### Load the catalog

Read the six catalog tables from the lakehouse into pandas. Missing tables are tolerated (they produce empty frames), so the run still completes for whichever signals are available.

In [ ]:
def load_delta(table_name):
    df = spark.read.format("delta").load('Tables/' + table_name)
    return df.toPandas()


models_df = load_delta("semantic_models")
tables_df = load_delta("semantic_model_tables")
columns_df = load_delta("semantic_model_columns")
relationships_df = load_delta("semantic_model_relationships")
measures_df = load_delta("semantic_model_measures")
datasources_df = load_delta("semantic_model_datasources")

if models_df.empty:
    raise ValueError(
        "No rows in the semantic_models table of the attached lakehouse. "
        "Run the TOM catalog notebook first."
    )

print(f"Models: {len(models_df)}")
print(
    f"Tables: {len(tables_df)} | Columns: {len(columns_df)} | "
    f"Measures: {len(measures_df)} | Relationships: {len(relationships_df)} | "
    f"Datasources: {len(datasources_df)}"
)

### Build model signatures

Each model is reduced to normalized sets (lowercased, whitespace-collapsed) for structural comparison, plus a text document (measure names + comment-stripped DAX) for embedding. Models with no measures fall back to their table and column names so the embedding document is never empty.

In [ ]:
def norm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()


def norm_dax(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value)
    text = re.sub(r"/\*.*?\*/", " ", text, flags=re.S)  # block comments
    text = re.sub(r"//.*", " ", text)  # line comments
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def model_label(sig):
    return f"{sig['workspace_name']} / {sig['model_name']}"


signatures = {}
for _, row in models_df.iterrows():
    model_id = str(row["model_id"])
    signatures[model_id] = {
        "model_id": model_id,
        "workspace_id": str(row.get("workspace_id", "")),
        "workspace_name": str(row.get("workspace_name", "")),
        "model_name": str(row.get("model_name", "")),
        "tables": set(),
        "columns": set(),
        "measure_names": set(),
        "measure_definitions": set(),
        "relationships": set(),
        "datasources": set(),
        "dax_docs": [],
    }


def sig_for(model_id):
    return signatures.get(str(model_id))


for _, row in tables_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["tables"].add(norm(row["table_name"]))

for _, row in columns_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["columns"].add(f"{norm(row['table_name'])}.{norm(row['column_name'])}")

for _, row in measures_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        measure_name = norm(row["measure_name"])
        measure_dax = norm_dax(row.get("expression"))
        sig["measure_names"].add(measure_name)
        # Name + DAX key so containment only credits measures whose logic also matches.
        sig["measure_definitions"].add(f"{measure_name} :: {measure_dax}")
        sig["dax_docs"].append(f"{measure_name} {measure_dax}".strip())

for _, row in relationships_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        key = (
            f"{norm(row['from_table'])}.{norm(row['from_column'])}"
            f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
        )
        sig["relationships"].add(key)

for _, row in datasources_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        conn = row.get("connection_string") or row.get("connection_details") or row.get("datasource_name")
        conn_norm = norm(conn)
        if conn_norm:
            sig["datasources"].add(conn_norm)

# Build the embedding document per model, with a structural fallback when no measures exist.
for sig in signatures.values():
    parts = list(sig["dax_docs"])
    if not parts:
        parts = sorted(sig["tables"]) + sorted(sig["columns"])
    sig["doc"] = " \n ".join(parts) if parts else (sig["model_name"] or sig["model_id"])

model_ids = list(signatures.keys())
print(f"Built signatures for {len(model_ids)} models.")


### Candidate pairs and structural similarity

With blocking enabled, only model pairs that share at least one table or measure name are scored, which avoids a full O(n²) comparison on large catalogs. Jaccard overlap is computed per signal for each candidate pair.

In [ ]:
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union else 0.0


def coverage(source_set, other_set):
    # Directional: the fraction of source_set's members that also appear in other_set.
    # Returns None when the signal is absent from the source, so it can be excluded from
    # the weighted average rather than counted as a spurious perfect match.
    if not source_set:
        return None
    return len(source_set & other_set) / len(source_set)


def weighted_containment(source_sig, other_sig, weights):
    # How completely source_sig is contained in other_sig: a weighted mean of the
    # per-signal coverages, normalized over whichever signals the source actually has.
    accumulated = 0.0
    total_weight = 0.0
    for signal, weight in weights.items():
        signal_coverage = coverage(source_sig[signal], other_sig[signal])
        if signal_coverage is None:
            continue
        accumulated += weight * signal_coverage
        total_weight += weight
    return accumulated / total_weight if total_weight else 0.0


def classify_containment(a_in_b, b_in_a, threshold):
    a_contained = a_in_b >= threshold
    b_contained = b_in_a >= threshold
    if a_contained and b_contained:
        return "equivalent"
    if b_contained:
        return "model_a_contains_model_b"
    if a_contained:
        return "model_b_contains_model_a"
    return "partial_overlap"


if ENABLE_BLOCKING:
    block_index = defaultdict(set)
    for model_id, sig in signatures.items():
        for table_name in sig["tables"]:
            block_index[("t", table_name)].add(model_id)
        for measure_name in sig["measure_names"]:
            block_index[("m", measure_name)].add(model_id)
    candidate_pairs = set()
    for group in block_index.values():
        if len(group) > 1:
            candidate_pairs.update(itertools.combinations(sorted(group), 2))
else:
    candidate_pairs = set(itertools.combinations(sorted(model_ids), 2))

print(f"Candidate pairs to score: {len(candidate_pairs)}")


### Semantic embeddings

Encode each model's document once as a TF-IDF vector over its DAX / name tokens (no PyTorch dependency), then derive cosine similarity from the L2-normalized vectors (a dot product) for each candidate pair.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF avoids the torch/transformers dependency chain (the Fabric runtime ships
# an older PyTorch than recent transformers require). Rows are L2-normalized, so
# cosine similarity stays a plain dot product for the downstream scoring step.
docs = [signatures[model_id]["doc"] for model_id in model_ids]
vectorizer = TfidfVectorizer(min_df=1, norm="l2")

embeddings = vectorizer.fit_transform(docs).toarray()
print(f"Encoded {len(docs)} model documents into {embeddings.shape[1]}-dim TF-IDF vectors.")
embedding_index = {model_id: idx for idx, model_id in enumerate(model_ids)}

### Score the models

Each candidate pair gets two independent scores:

- **Composite score** — a weighted blend of the six similarity signals that answers *"how alike are these two models overall?"*. It is symmetric, drives the duplicate / similar / distinct tiers, and duplicate-tier pairs are merged into clusters with union-find.
- **Containment score** — a directional measure that answers *"does one model contain everything in the other?"*. It is the stronger of the two directional coverages (A-in-B and B-in-A), and `containment_relationship` records which model contains which. A small model fully absorbed by a much larger one scores high here even when the composite score is only moderate.

Both scores are written to every pair row so you can filter on either one.


In [ ]:
weight_sum = sum(SIMILARITY_WEIGHTS.values())
pair_rows = []

for model_id_a, model_id_b in candidate_pairs:
    sig_a = signatures[model_id_a]
    sig_b = signatures[model_id_b]

    j_tables = jaccard(sig_a["tables"], sig_b["tables"])
    j_columns = jaccard(sig_a["columns"], sig_b["columns"])
    j_measures = jaccard(sig_a["measure_names"], sig_b["measure_names"])
    j_relationships = jaccard(sig_a["relationships"], sig_b["relationships"])
    j_datasources = jaccard(sig_a["datasources"], sig_b["datasources"])
    cosine = float(
        np.dot(embeddings[embedding_index[model_id_a]], embeddings[embedding_index[model_id_b]])
    )
    cosine = max(0.0, min(1.0, cosine))

    composite = (
        SIMILARITY_WEIGHTS["tables"] * j_tables
        + SIMILARITY_WEIGHTS["columns"] * j_columns
        + SIMILARITY_WEIGHTS["measure_names"] * j_measures
        + SIMILARITY_WEIGHTS["measure_dax_embedding"] * cosine
        + SIMILARITY_WEIGHTS["relationships"] * j_relationships
        + SIMILARITY_WEIGHTS["datasources"] * j_datasources
    ) / weight_sum

    # Directional containment: how much of each model is absorbed by the other.
    a_in_b = weighted_containment(sig_a, sig_b, CONTAINMENT_WEIGHTS)
    b_in_a = weighted_containment(sig_b, sig_a, CONTAINMENT_WEIGHTS)
    containment_score = max(a_in_b, b_in_a)
    containment_relationship = classify_containment(a_in_b, b_in_a, CONTAINMENT_THRESHOLD)

    if composite >= DUPLICATE_THRESHOLD:
        tier = "duplicate"
    elif composite >= SIMILAR_THRESHOLD:
        tier = "similar"
    else:
        tier = "distinct"

    pair_rows.append({
        "model_id_a": model_id_a,
        "model_a": model_label(sig_a),
        "workspace_a": sig_a["workspace_name"],
        "model_id_b": model_id_b,
        "model_b": model_label(sig_b),
        "workspace_b": sig_b["workspace_name"],
        "same_model_name": norm(sig_a["model_name"]) == norm(sig_b["model_name"]),
        "cross_workspace": sig_a["workspace_id"] != sig_b["workspace_id"],
        "jaccard_tables": round(j_tables, 4),
        "jaccard_columns": round(j_columns, 4),
        "jaccard_measure_names": round(j_measures, 4),
        "jaccard_relationships": round(j_relationships, 4),
        "jaccard_datasources": round(j_datasources, 4),
        "dax_embedding_cosine": round(cosine, 4),
        "composite_score": round(composite, 4),
        "containment_score": round(containment_score, 4),
        "containment_relationship": containment_relationship,
        "model_a_in_model_b": round(a_in_b, 4),
        "model_b_in_model_a": round(b_in_a, 4),
        "tier": tier,
    })

pairs_df = pd.DataFrame(pair_rows)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values("composite_score", ascending=False).reset_index(drop=True)

# Union-find clustering over duplicate-tier pairs.
parent = {model_id: model_id for model_id in model_ids}


def find(node):
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = parent[node]
    return node


def union(node_a, node_b):
    root_a, root_b = find(node_a), find(node_b)
    if root_a != root_b:
        parent[root_a] = root_b


if not pairs_df.empty:
    for _, row in pairs_df[pairs_df["tier"] == "duplicate"].iterrows():
        union(row["model_id_a"], row["model_id_b"])

cluster_members = defaultdict(list)
for model_id in model_ids:
    cluster_members[find(model_id)].append(model_id)

cluster_rows = []
cluster_number = 0
for members in cluster_members.values():
    if len(members) > 1:
        cluster_number += 1
        for model_id in members:
            sig = signatures[model_id]
            cluster_rows.append({
                "cluster_id": cluster_number,
                "cluster_size": len(members),
                "model_id": model_id,
                "model": model_label(sig),
                "workspace_name": sig["workspace_name"],
                "model_name": sig["model_name"],
            })

clusters_df = pd.DataFrame(cluster_rows)

# Per-model signature summary.
signature_rows = []
for sig in signatures.values():
    signature_rows.append({
        "model_id": sig["model_id"],
        "workspace_name": sig["workspace_name"],
        "model_name": sig["model_name"],
        "table_count": len(sig["tables"]),
        "column_count": len(sig["columns"]),
        "measure_count": len(sig["measure_names"]),
        "relationship_count": len(sig["relationships"]),
        "datasource_count": len(sig["datasources"]),
    })
signatures_df = pd.DataFrame(signature_rows)

duplicate_count = int((pairs_df["tier"] == "duplicate").sum()) if not pairs_df.empty else 0
similar_count = int((pairs_df["tier"] == "similar").sum()) if not pairs_df.empty else 0
containment_count = (
    int((pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD).sum()) if not pairs_df.empty else 0
)
print(f"Duplicate pairs: {duplicate_count} | Similar pairs: {similar_count} | Containment pairs: {containment_count}")
print(f"Duplicate clusters: {cluster_number}")


## Results

An interactive summary of this run. Use the tabs to move between the headline **Overview**, the full list of **Duplicates** and near-duplicates, **Containment** (subset) relationships, duplicate **Clusters**, and the similarity **Matrix**. Select any row to reveal the evidence behind its score.

In [ ]:
# Interactive results app: a single navigable in-notebook experience rendered via displayHTML.
# All data is embedded; tab navigation, search and drill-down run client-side (no dataframes shown).
import json
from datetime import datetime, timezone


def _round(value):
    try:
        return round(float(value), 4)
    except (TypeError, ValueError):
        return None


total_models = int(len(signatures_df)) if not signatures_df.empty else len(signatures)

if not pairs_df.empty:
    _flagged = pairs_df[pairs_df["tier"].isin(["duplicate", "similar"])].sort_values("composite_score", ascending=False)
    _contained = pairs_df[pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD].sort_values("containment_score", ascending=False)
else:
    _flagged = pairs_df
    _contained = pairs_df


def _pair_obj(row):
    return {
        "tier": row["tier"],
        "composite": _round(row["composite_score"]),
        "containment": _round(row["containment_score"]),
        "relationship": row.get("containment_relationship", ""),
        "modelA": str(row["model_a"]), "workspaceA": str(row["workspace_a"]),
        "modelB": str(row["model_b"]), "workspaceB": str(row["workspace_b"]),
        "aInB": _round(row["model_a_in_model_b"]), "bInA": _round(row["model_b_in_model_a"]),
        "crossWorkspace": bool(row.get("cross_workspace", False)),
        "sameName": bool(row.get("same_model_name", False)),
        "jaccard": {
            "tables": _round(row["jaccard_tables"]), "columns": _round(row["jaccard_columns"]),
            "measures": _round(row["jaccard_measure_names"]), "relationships": _round(row["jaccard_relationships"]),
            "datasources": _round(row["jaccard_datasources"]),
        },
        "daxCosine": _round(row["dax_embedding_cosine"]),
    }


_pairs_payload = [_pair_obj(r) for _, r in _flagged.iterrows()]
_containment_payload = [_pair_obj(r) for _, r in _contained.iterrows()]

_clusters_payload = []
if not clusters_df.empty:
    for _cid, _grp in clusters_df.sort_values(["cluster_id", "model"]).groupby("cluster_id"):
        _clusters_payload.append({
            "id": int(_cid),
            "size": int(_grp["cluster_size"].iloc[0]),
            "members": [{"model": str(m["model_name"]), "workspace": str(m["workspace_name"])} for _, m in _grp.iterrows()],
        })

_matrix_payload = {"labels": [], "workspaces": [], "z": []}
if not _flagged.empty:
    _ids = sorted(
        set(_flagged["model_id_a"]) | set(_flagged["model_id_b"]),
        key=lambda mid: (signatures[mid]["model_name"], signatures[mid]["workspace_name"]),
    )
    _pos = {mid: index for index, mid in enumerate(_ids)}
    _n = len(_ids)
    _z = np.eye(_n)
    for _, row in _flagged.iterrows():
        i, j = _pos[row["model_id_a"]], _pos[row["model_id_b"]]
        _z[i, j] = _z[j, i] = float(row["composite_score"])
    _matrix_payload = {
        "labels": [signatures[mid]["model_name"] for mid in _ids],
        "workspaces": [signatures[mid]["workspace_name"] for mid in _ids],
        "z": [[_round(v) for v in r] for r in _z.tolist()],
    }

_app_data = {
    "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "summary": {
        "models": int(total_models),
        "duplicatePairs": int(duplicate_count),
        "similarPairs": int(similar_count),
        "containmentCandidates": int(containment_count),
        "clusters": int(cluster_number),
    },
    "thresholds": {
        "duplicate": _round(DUPLICATE_THRESHOLD),
        "similar": _round(SIMILAR_THRESHOLD),
        "containment": _round(CONTAINMENT_THRESHOLD),
    },
    "pairs": _pairs_payload,
    "containment": _containment_payload,
    "clusters": _clusters_payload,
    "matrix": _matrix_payload,
}

_APP_TEMPLATE = r"""<style>
#sms-app{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;color:#1d1d1f;background:#f5f5f7;border-radius:20px;max-width:1120px;margin:8px auto;overflow:hidden;-webkit-font-smoothing:antialiased;}
#sms-app *{box-sizing:border-box;}
#sms-app .apphead{display:flex;align-items:flex-end;justify-content:space-between;padding:24px 28px 16px;}
#sms-app .eyebrow{font-size:12px;font-weight:600;letter-spacing:.07em;text-transform:uppercase;color:#8a8a8e;}
#sms-app .apptitle{font-size:22px;font-weight:600;letter-spacing:-.01em;margin-top:3px;}
#sms-app .gen{font-size:12px;color:#a1a1a6;}
#sms-app .tabs{display:flex;gap:4px;padding:4px;margin:0 28px;background:#e9e9ec;border-radius:12px;overflow-x:auto;}
#sms-app .tab{flex:0 0 auto;border:0;background:transparent;color:#4a4a4f;font:inherit;font-size:13px;font-weight:600;padding:8px 16px;border-radius:9px;cursor:pointer;display:flex;align-items:center;gap:7px;white-space:nowrap;}
#sms-app .tab.active{background:#fff;color:#0071e3;box-shadow:0 1px 3px rgba(0,0,0,.1);}
#sms-app .tcount{font-size:11px;font-weight:600;background:rgba(0,0,0,.06);color:#6e6e73;border-radius:999px;padding:1px 7px;}
#sms-app .tab.active .tcount{background:#eaf3ff;color:#0071e3;}
#sms-app #sms-view{padding:22px 28px 26px;}
#sms-app .head2{font-size:19px;font-weight:600;letter-spacing:-.01em;line-height:1.32;margin-bottom:18px;}
#sms-app .kpis{display:flex;flex-wrap:wrap;gap:12px;}
#sms-app .kpi{flex:1 1 130px;background:#fff;border-radius:14px;padding:16px 18px;box-shadow:0 1px 3px rgba(0,0,0,.06);}
#sms-app .kpi .n{font-size:30px;font-weight:600;letter-spacing:-.02em;line-height:1;}
#sms-app .kpi .l{font-size:12px;color:#6e6e73;margin-top:6px;}
#sms-app .kpi.accent .n{color:#0071e3;}
#sms-app .kpi.warn .n{color:#c9330a;}
#sms-app .secrow{display:flex;align-items:center;justify-content:space-between;margin:24px 0 12px;}
#sms-app .sec{font-size:16px;font-weight:600;margin:0;}
#sms-app .btnlink{border:0;background:transparent;color:#0071e3;font:inherit;font-size:13px;font-weight:600;cursor:pointer;padding:0;}
#sms-app .toolbar{display:flex;gap:10px;align-items:center;margin-bottom:14px;flex-wrap:wrap;}
#sms-app .search{flex:1 1 240px;min-width:180px;border:1px solid #e0e0e3;background:#fff;border-radius:10px;padding:9px 13px;font:inherit;font-size:13px;color:#1d1d1f;}
#sms-app .search:focus{outline:none;border-color:#0071e3;box-shadow:0 0 0 3px rgba(0,113,227,.12);}
#sms-app .chips{display:flex;gap:6px;}
#sms-app .fchip{border:1px solid #e0e0e3;background:#fff;color:#4a4a4f;font:inherit;font-size:12.5px;font-weight:600;padding:8px 13px;border-radius:999px;cursor:pointer;}
#sms-app .fchip.on{background:#0071e3;border-color:#0071e3;color:#fff;}
#sms-app .plist,#sms-app .clist{display:flex;flex-direction:column;gap:10px;}
#sms-app .pcard{background:#fff;border-radius:14px;box-shadow:0 1px 3px rgba(0,0,0,.06);overflow:hidden;}
#sms-app .phead{display:flex;align-items:center;gap:16px;padding:14px 16px;cursor:pointer;}
#sms-app .pinfo{flex:1 1 auto;min-width:0;}
#sms-app .ptop{display:flex;align-items:center;gap:6px;margin-bottom:7px;}
#sms-app .pnames{font-size:14px;line-height:1.5;}
#sms-app .mname{font-weight:600;}
#sms-app .wname{font-size:12px;color:#8a8a8e;}
#sms-app .vs{color:#c7c7cc;margin:0 7px;}
#sms-app .pscore{flex:0 0 170px;}
#sms-app .track{height:7px;background:#ececee;border-radius:6px;overflow:hidden;}
#sms-app .fill{height:100%;border-radius:6px;background:#0071e3;}
#sms-app .fill.cov{background:#0058c9;}
#sms-app .fill.mini{background:#8e8e93;}
#sms-app .scoreval{font-size:11.5px;color:#6e6e73;margin-top:4px;}
#sms-app .chev{flex:0 0 auto;color:#c7c7cc;font-size:20px;line-height:1;transition:transform .15s ease;}
#sms-app .pcard.open .chev{transform:rotate(90deg);}
#sms-app .pdetail{display:none;padding:0 16px 16px;}
#sms-app .pcard.open .pdetail{display:block;}
#sms-app .verdict{font-size:13.5px;color:#3a3a3c;border-top:1px solid #f0f0f2;padding-top:12px;margin-bottom:12px;}
#sms-app .evid{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:10px 16px;}
#sms-app .ev{font-size:12px;}
#sms-app .evl{color:#8a8a8e;margin-bottom:5px;}
#sms-app .ev .track{height:5px;}
#sms-app .evv{font-size:11.5px;color:#3a3a3c;margin-top:3px;font-weight:600;}
#sms-app .pill{display:inline-block;font-size:11px;font-weight:600;padding:2px 9px;border-radius:999px;}
#sms-app .pill.dup{background:#ffe9e3;color:#c9330a;}
#sms-app .pill.sim{background:#fff2df;color:#96590a;}
#sms-app .chip{display:inline-block;font-size:10.5px;font-weight:500;padding:2px 8px;border-radius:999px;background:#eef1f6;color:#5b5b60;}
#sms-app .chip.xws{background:#efe9fc;color:#6b3fd4;}
#sms-app .grid2{display:grid;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:12px;}
#sms-app .card{background:#fff;border-radius:14px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,.06);}
#sms-app .crow{display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;}
#sms-app .badge{font-size:11px;font-weight:600;color:#0071e3;background:#eef4ff;border-radius:999px;padding:3px 10px;}
#sms-app .members{list-style:none;margin:0;padding:0;}
#sms-app .members li{display:flex;justify-content:space-between;align-items:baseline;padding:7px 0;border-top:1px solid #f2f2f4;}
#sms-app .members li:first-child{border-top:0;}
#sms-app .members .mm{font-size:13.5px;font-weight:600;}
#sms-app .members .w{font-size:12px;color:#8a8a8e;}
#sms-app .empty{background:#fff;border-radius:14px;padding:28px;text-align:center;color:#6e6e73;font-size:14px;box-shadow:0 1px 3px rgba(0,0,0,.06);}
#sms-app .mwrap{overflow:auto;background:#fff;border-radius:14px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,.06);}
#sms-app .mgrid{display:grid;gap:2px;align-items:center;}
#sms-app .mcolh{font-size:11px;color:#8a8a8e;text-align:center;font-weight:600;}
#sms-app .mrowh{font-size:12px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;padding-right:8px;}
#sms-app .mrowh .mnum{display:inline-block;width:20px;color:#a1a1a6;font-weight:600;}
#sms-app .mcell{width:30px;height:30px;border-radius:5px;}
#sms-app .mlegend{display:flex;align-items:center;gap:8px;margin-top:14px;font-size:11.5px;color:#8a8a8e;}
#sms-app .mbar{height:8px;width:120px;border-radius:6px;background:linear-gradient(90deg,hsl(211,88%,94%),hsl(211,88%,36%));}
#sms-app .foot{font-size:11.5px;color:#a1a1a6;padding:0 28px 22px;line-height:1.6;}
</style>
<div id="sms-app"></div>
<script>
(function(){
  var DATA = __APP_DATA__;
  var root = document.getElementById('sms-app');
  if(!root){ return; }
  var S = DATA.summary, T = DATA.thresholds;
  var state = { tab:'overview', pairSearch:'', pairTier:'all', contSearch:'' };

  function esc(s){ return String(s==null?'':s).replace(/[&<>"']/g, function(c){ return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]; }); }
  function num(v){ return (v==null)?'&mdash;':(Math.round(v*100)/100).toFixed(2); }
  function bar(v, cls){ var w=Math.max(0,Math.min(1,v||0))*100; return '<div class="track"><div class="fill '+(cls||'')+'" style="width:'+w.toFixed(1)+'%"></div></div>'; }

  var tabs = [
    {id:'overview', label:'Overview'},
    {id:'pairs', label:'Duplicates', count:DATA.pairs.length, show:DATA.pairs.length>0},
    {id:'containment', label:'Containment', count:DATA.containment.length, show:DATA.containment.length>0},
    {id:'clusters', label:'Clusters', count:DATA.clusters.length, show:DATA.clusters.length>0},
    {id:'matrix', label:'Matrix', show:DATA.matrix.labels.length>1}
  ].filter(function(t){ return t.show!==false; });

  function headline(){
    if(DATA.pairs.length===0 && DATA.clusters.length===0 && DATA.containment.length===0){
      return 'No duplicates or near-duplicates found across '+S.models+' models at the current thresholds.';
    }
    var parts=[];
    if(S.duplicatePairs){ parts.push(S.duplicatePairs+' duplicate'); }
    if(S.similarPairs){ parts.push(S.similarPairs+' near-duplicate'); }
    var lead = parts.length ? parts.join(' and ')+' pair'+(((S.duplicatePairs+S.similarPairs)!==1)?'s':'') : 'overlapping models';
    var tail='';
    if(S.containmentCandidates){ tail += ', plus '+S.containmentCandidates+' subset relationship'+((S.containmentCandidates!==1)?'s':''); }
    if(S.clusters){ tail += ', forming '+S.clusters+' duplicate cluster'+((S.clusters!==1)?'s':''); }
    return 'Across '+S.models+' models, found '+lead+tail+'.';
  }

  function verdict(p){
    if(p.tier==='duplicate' && p.crossWorkspace){ return 'Very likely the same model in two workspaces &mdash; consolidate to a single source.'; }
    if(p.tier==='duplicate' && p.sameName){ return 'Duplicate models sharing a name &mdash; consolidate.'; }
    if(p.tier==='duplicate'){ return 'Near-identical structure &mdash; a strong consolidation candidate.'; }
    if((p.relationship==='model_a_contains_model_b'||p.relationship==='model_b_contains_model_a') && p.containment>=T.containment){ return 'One model is effectively a subset of the other &mdash; consider retiring the smaller one.'; }
    return 'Substantial overlap &mdash; review for shared logic or consolidation.';
  }

  function evItem(label, v){ return '<div class="ev"><div class="evl">'+label+'</div>'+bar(v,'mini')+'<div class="evv">'+num(v)+'</div></div>'; }

  function pairCard(p){
    var t = (p.tier==='duplicate')?'dup':'sim';
    var tl = (p.tier==='duplicate')?'Duplicate':'Similar';
    var chips = (p.crossWorkspace?'<span class="chip xws">cross-workspace</span>':'') + (p.sameName?'<span class="chip">same name</span>':'');
    return '<div class="pcard"><div class="phead"><div class="pinfo">'+
      '<div class="ptop"><span class="pill '+t+'">'+tl+'</span>'+chips+'</div>'+
      '<div class="pnames"><span class="mname">'+esc(p.modelA)+'</span> <span class="wname">&middot; '+esc(p.workspaceA)+'</span> <span class="vs">&harr;</span> <span class="mname">'+esc(p.modelB)+'</span> <span class="wname">&middot; '+esc(p.workspaceB)+'</span></div>'+
      '</div><div class="pscore">'+bar(p.composite)+'<div class="scoreval">'+num(p.composite)+' composite</div></div><div class="chev">&rsaquo;</div></div>'+
      '<div class="pdetail"><div class="verdict">'+verdict(p)+'</div><div class="evid">'+
        evItem('Tables', p.jaccard.tables)+evItem('Columns', p.jaccard.columns)+evItem('Measures', p.jaccard.measures)+
        evItem('Relationships', p.jaccard.relationships)+evItem('Data sources', p.jaccard.datasources)+evItem('DAX text', p.daxCosine)+evItem('Containment', p.containment)+
      '</div></div></div>';
  }

  function contCard(p){
    var eqv = (p.relationship==='equivalent');
    var a,b,cov;
    if(eqv){ a=p.modelA; b=p.modelB; cov=Math.max(p.aInB||0, p.bInA||0); }
    else if((p.aInB||0) >= (p.bInA||0)){ a=p.modelA; b=p.modelB; cov=p.aInB; }
    else { a=p.modelB; b=p.modelA; cov=p.bInA; }
    var sym = eqv ? '&equiv;' : '&sub;';
    var note = eqv ? 'Equivalent &mdash; each model contains the other.' : (esc(a)+' is contained in '+esc(b)+'.');
    return '<div class="pcard"><div class="phead"><div class="pinfo">'+
      '<div class="pnames"><span class="mname">'+esc(a)+'</span> <span class="vs">'+sym+'</span> <span class="mname">'+esc(b)+'</span></div>'+
      '<div class="wname" style="margin-top:5px;">'+note+'</div></div>'+
      '<div class="pscore">'+bar(cov,'cov')+'<div class="scoreval">'+num(cov)+' coverage</div></div><div class="chev">&rsaquo;</div></div>'+
      '<div class="pdetail"><div class="evid">'+evItem('A in B', p.aInB)+evItem('B in A', p.bInA)+evItem('Composite', p.composite)+evItem('Containment', p.containment)+'</div></div></div>';
  }

  function filteredPairs(){
    var q = state.pairSearch.trim().toLowerCase();
    return DATA.pairs.filter(function(p){
      if(state.pairTier!=='all' && p.tier!==state.pairTier){ return false; }
      if(!q){ return true; }
      return (p.modelA+' '+p.workspaceA+' '+p.modelB+' '+p.workspaceB).toLowerCase().indexOf(q)>=0;
    });
  }
  function filteredCont(){
    var q = state.contSearch.trim().toLowerCase();
    return DATA.containment.filter(function(p){
      if(!q){ return true; }
      return (p.modelA+' '+p.workspaceA+' '+p.modelB+' '+p.workspaceB).toLowerCase().indexOf(q)>=0;
    });
  }

  function overviewHTML(){
    var kpis = [
      ['accent', S.models, 'Models scanned'],
      [S.duplicatePairs?'warn':'', S.duplicatePairs, 'Duplicate pairs'],
      ['', S.similarPairs, 'Similar pairs'],
      ['', S.containmentCandidates, 'Containment'],
      ['', S.clusters, 'Clusters']
    ].map(function(c){ return '<div class="kpi '+c[0]+'"><div class="n">'+c[1]+'</div><div class="l">'+c[2]+'</div></div>'; }).join('');
    var body;
    if(DATA.pairs.length>0){
      var top = DATA.pairs.slice(0,5).map(pairCard).join('');
      var seeAll = (DATA.pairs.length>5) ? '<button class="btnlink" data-goto="pairs">See all '+DATA.pairs.length+' &rsaquo;</button>' : '';
      body = '<div class="secrow"><h3 class="sec">Top consolidation candidates</h3>'+seeAll+'</div><div class="plist">'+top+'</div>';
    } else {
      body = '<div class="empty">No duplicates or near-duplicates at the current thresholds. Lower DUPLICATE_THRESHOLD / SIMILAR_THRESHOLD in Parameters to widen the search.</div>';
    }
    return '<div class="head2">'+esc(headline())+'</div><div class="kpis">'+kpis+'</div>'+body;
  }

  function pairsHTML(){
    return '<div class="toolbar"><input class="search" type="text" placeholder="Search models or workspaces" value="'+esc(state.pairSearch)+'">'+
      '<div class="chips">'+
        '<button class="fchip'+(state.pairTier==='all'?' on':'')+'" data-tier="all">All</button>'+
        '<button class="fchip'+(state.pairTier==='duplicate'?' on':'')+'" data-tier="duplicate">Duplicates</button>'+
        '<button class="fchip'+(state.pairTier==='similar'?' on':'')+'" data-tier="similar">Similar</button>'+
      '</div></div><div class="plist"></div>';
  }
  function contHTML(){
    return '<div class="toolbar"><input class="search" type="text" placeholder="Search models or workspaces" value="'+esc(state.contSearch)+'"></div><div class="clist"></div>';
  }
  function clustersHTML(){
    if(DATA.clusters.length===0){ return '<div class="empty">No duplicate clusters were found.</div>'; }
    return '<div class="grid2">'+ DATA.clusters.map(function(c){
      return '<div class="card"><div class="crow"><div class="mname">Cluster '+c.id+'</div><span class="badge">'+c.size+' models</span></div><ul class="members">'+
        c.members.map(function(m){ return '<li><span class="mm">'+esc(m.model)+'</span><span class="w">'+esc(m.workspace)+'</span></li>'; }).join('')+
      '</ul></div>';
    }).join('') +'</div>';
  }
  function cellColor(v){ if(v==null){ return '#f2f2f4'; } var L = 94 - Math.max(0,Math.min(1,v))*58; return 'hsl(211,88%,'+L.toFixed(0)+'%)'; }
  function matrixHTML(){
    var m = DATA.matrix, n = m.labels.length;
    var h = '<div class="mwrap"><div class="mgrid" style="grid-template-columns:200px repeat('+n+',30px);">';
    h += '<div></div>';
    for(var j=0;j<n;j++){ h += '<div class="mcolh" title="'+esc(m.labels[j])+' &middot; '+esc(m.workspaces[j])+'">'+(j+1)+'</div>'; }
    for(var i=0;i<n;i++){
      h += '<div class="mrowh" title="'+esc(m.labels[i])+' &middot; '+esc(m.workspaces[i])+'"><span class="mnum">'+(i+1)+'</span>'+esc(m.labels[i])+'</div>';
      for(var k=0;k<n;k++){
        var v = m.z[i][k];
        h += '<div class="mcell" title="'+esc(m.labels[i])+' &harr; '+esc(m.labels[k])+': '+num(v)+'" style="background:'+cellColor(v)+'"></div>';
      }
    }
    h += '</div><div class="mlegend">Composite similarity<span class="mbar"></span>low to high</div></div>';
    return h;
  }

  function footHTML(){
    return '<div class="foot">Duplicate &ge; '+T.duplicate+' composite &middot; Similar &ge; '+T.similar+' &middot; Containment &ge; '+T.containment+' coverage. Metadata-based (tables, columns, measures &amp; DAX, relationships, data sources) &mdash; confirm intent before consolidating.</div>';
  }
  function tabsHTML(){
    return '<div class="tabs">'+ tabs.map(function(t){
      return '<button class="tab'+(t.id===state.tab?' active':'')+'" data-tab="'+t.id+'">'+t.label+(t.count!=null?' <span class="tcount">'+t.count+'</span>':'')+'</button>';
    }).join('') +'</div>';
  }
  function headerHTML(){
    return '<div class="apphead"><div><div class="eyebrow">Semantic Model Similarity</div><div class="apptitle">Duplicate &amp; overlap explorer</div></div><div class="gen">'+esc(DATA.generatedAt)+'</div></div>';
  }

  function updatePairList(){
    var list = root.querySelector('.plist'); if(!list){ return; }
    var items = filteredPairs();
    list.innerHTML = items.length ? items.map(pairCard).join('') : '<div class="empty">No pairs match your search.</div>';
  }
  function updateContList(){
    var list = root.querySelector('.clist'); if(!list){ return; }
    var items = filteredCont();
    list.innerHTML = items.length ? items.map(contCard).join('') : '<div class="empty">No containment candidates match your search.</div>';
  }

  function renderView(){
    var v = root.querySelector('#sms-view'); if(!v){ return; }
    if(state.tab==='overview'){ v.innerHTML = overviewHTML(); }
    else if(state.tab==='pairs'){ v.innerHTML = pairsHTML(); updatePairList(); wirePairs(v); }
    else if(state.tab==='containment'){ v.innerHTML = contHTML(); updateContList(); wireCont(v); }
    else if(state.tab==='clusters'){ v.innerHTML = clustersHTML(); }
    else if(state.tab==='matrix'){ v.innerHTML = matrixHTML(); }
  }
  function wirePairs(v){
    var s = v.querySelector('.search');
    if(s){ s.addEventListener('input', function(e){ state.pairSearch = e.target.value; updatePairList(); }); }
    v.querySelectorAll('.fchip').forEach(function(b){ b.addEventListener('click', function(){ state.pairTier = b.dataset.tier; v.querySelectorAll('.fchip').forEach(function(x){ x.classList.toggle('on', x===b); }); updatePairList(); }); });
  }
  function wireCont(v){
    var s = v.querySelector('.search');
    if(s){ s.addEventListener('input', function(e){ state.contSearch = e.target.value; updateContList(); }); }
  }
  function setTab(id){ state.tab=id; root.querySelectorAll('.tab').forEach(function(b){ b.classList.toggle('active', b.dataset.tab===id); }); renderView(); }

  function render(){
    root.innerHTML = headerHTML()+tabsHTML()+'<div id="sms-view"></div>'+footHTML();
    root.querySelectorAll('.tab').forEach(function(b){ b.addEventListener('click', function(){ setTab(b.dataset.tab); }); });
    var view = root.querySelector('#sms-view');
    view.addEventListener('click', function(e){
      var goto = e.target.closest('[data-goto]');
      if(goto){ setTab(goto.dataset.goto); return; }
      var head = e.target.closest('.phead');
      if(head){ head.parentElement.classList.toggle('open'); }
    });
    renderView();
  }

  render();
})();
</script>"""

_app_json = json.dumps(_app_data).replace("<", "\\u003c")
displayHTML(_APP_TEMPLATE.replace("__APP_DATA__", _app_json))


## Save results to the lakehouse

Write the per-model signature summary, the full pairwise scores, and the duplicate clusters as Delta tables so the analysis is queryable outside this notebook.

In [ ]:
similarity_outputs = {
    "semantic_model_signatures": signatures_df,
    "semantic_model_similarity_pairs": pairs_df,
    "semantic_model_duplicate_clusters": clusters_df,
}

for table_name, frame in similarity_outputs.items():
    if frame.empty:
        print(f"Skipped {table_name}: no rows")
        continue
    spark.createDataFrame(frame).write.format("delta").mode(WRITE_MODE).option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")

## Next steps

- Start with the **Overview** and **Clusters** tabs — the strongest consolidation candidates. A `cross-workspace` tag often means the same model was copied between workspaces.
- Open the **Containment** tab to find subset relationships: one model holding everything in another (plus more) is a superseding model or an extract that may be retired. `semantic_model_similarity_pairs` carries both `composite_score` and `containment_score`, so you can filter on either independently (for example, high containment with only moderate similarity = a small model absorbed by a much larger one).
- Similarity and containment are metadata-based. Before acting on a pair, confirm the models truly serve the same purpose; they do not compare report layouts, row-level data, refresh history, or security roles.
- If the tiers look too strict or too loose, calibrate `DUPLICATE_THRESHOLD`, `SIMILAR_THRESHOLD`, `CONTAINMENT_THRESHOLD`, `SIMILARITY_WEIGHTS`, and `CONTAINMENT_WEIGHTS` in the Parameters cell against a few known pairs, then re-run.
- The three output tables (`semantic_model_signatures`, `semantic_model_similarity_pairs`, `semantic_model_duplicate_clusters`) are in the attached lakehouse for querying outside this notebook.